In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, callbacks

import gc

from preprocessor import TextEncoder
import pickle

In [15]:
# Load pre-cleaned data
cars_clean=pd.read_pickle('cars_clean.pkl')

In [20]:
cars_clean.loc[cars_clean.model == 'Fiesta'].variant.unique()

array(['Active', 'Active 1', 'Active Edition', 'Active Vignale',
       'Active X', 'Active X Edition', 'Centura', 'Chequered Flag',
       'ECOnetic', 'EcoBoost', 'EcoBoost MHEV', 'Equipe', 'Finesse',
       'Finesse II', 'Firefly', 'Flame', 'Flight', 'Freedom', 'Freestyle',
       'Ghia', 'L', 'LX', 'Metal', 'Quartz', 'RS', 'S', 'ST',
       'ST Edition', 'ST Performance Edition', 'ST-1', 'ST-2', 'ST-200',
       'ST-3', 'ST-500', 'ST-Line', 'ST-Line Black Edition',
       'ST-Line Edition', 'ST-Line Red Edition', 'ST-Line Vignale',
       'ST-Line X', 'ST-Line X Edition', 'Silver', 'Studio', 'Style',
       'Style Climate', 'T', 'TD', 'TDCi', 'Ti-VCT', 'Titanium',
       'Titanium Vignale', 'Titanium X', 'Trend', 'Vignale',
       'Vignale Edition', 'XR2i', 'Zetec', 'Zetec Black Edition',
       'Zetec Blue Edition', 'Zetec Climate', 'Zetec LX', 'Zetec S',
       'Zetec S Black Edition', 'Zetec S Red Edition',
       'Zetec White Edition', 'i'], dtype=object)

In [3]:
# ==========================================
# 1. SETUP & DATA PREP
# ==========================================


print("--- Preparing Data ---")

# 1. Shuffle & Split
shuffled_df = cars_clean.sample(frac=1, random_state=42).reset_index(drop=True)

train_size = int(0.8 * len(shuffled_df))
val_size = int(0.1 * len(shuffled_df))

train_df = shuffled_df.iloc[:train_size]
val_df = shuffled_df.iloc[train_size:train_size+val_size]
test_df = shuffled_df.iloc[train_size+val_size:]

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

--- Preparing Data ---
Train: 610848 | Val: 76356 | Test: 76356


In [4]:
# ==========================================
# 2. VOCABULARY & ENCODING  (Depends on the training data)
# ==========================================
print("--- Building Vocabularies ---")

# Set relevant features
cat_features = ['make', 'model', 'variant', 'fuel_type']
num_features = ['miles', 'year']

# Keep only relevant columns for modeling
train_df = train_df[cat_features + num_features + ['car_price']]
val_df = val_df[cat_features + num_features + ['car_price']]
test_df = test_df[cat_features + num_features + ['car_price']]

# Make an encoder
encoder = TextEncoder()

# Fit the encoder on the training data (builds vocabularies for the categorical features set above)
encoder.fit(cars_clean, cat_features)

# Pickle the encoder function for later predictions
with open('encoder.pkl', 'wb') as f:
    pickle.dump(encoder, f)


# Encode categorical features and combine with numerical features for model input
train_inputs = encoder.transform_df(train_df) | {'num_in': train_df[num_features].values.astype('float32')}
val_inputs = encoder.transform_df(val_df) | {'num_in': val_df[num_features].values.astype('float32')}
test_inputs = encoder.transform_df(test_df) | {'num_in': test_df[num_features].values.astype('float32')}


# We train the model to predict log(price), not raw price.
train_targets_log = np.log1p(train_df['car_price'].values.astype('float32'))
val_targets_log = np.log1p(val_df['car_price'].values.astype('float32'))
test_targets_raw = test_df['car_price'].values.astype('float32') # Keep raw for final eval



# Clean up RAM
del shuffled_df, cars_clean
gc.collect()



--- Building Vocabularies ---


5

In [11]:

# ==========================================
# 3. MODEL ARCHITECTURE
# ==========================================
print("--- Building Model ---")

# --- Branch A: Make ---
in_make = layers.Input(shape=(1,), name='make')
emb_make = layers.Flatten()(layers.Embedding(len(encoder.vocabs['make'])+1, 10)(in_make))

# --- Branch B: Model ---
in_model = layers.Input(shape=(1,), name='model')
emb_model = layers.Flatten()(layers.Embedding(len(encoder.vocabs['model'])+1, 15)(in_model))

# --- Branch C: Variant --- 
in_variant = layers.Input(shape=(1,), name='variant')
emb_variant = layers.Flatten()(layers.Embedding(len(encoder.vocabs['variant'])+1, 10)(in_variant))

# --- Branch D: Fuel Type --- 
in_fuel = layers.Input(shape=(1,), name='fuel_type')
# Few categories (Petrol, Diesel, EV...), so small vector (3-5) is fine
emb_fuel = layers.Flatten()(layers.Embedding(len(encoder.vocabs['fuel_type'])+1, 5)(in_fuel))

# --- Branch E: Numericals ---
in_num = layers.Input(shape=(2,), name='num_in')
normalizer = layers.Normalization(axis=-1)
normalizer.adapt(train_inputs['num_in'])
norm_num = normalizer(in_num)


# --- Merge All 5 Branches ---
merged = layers.Concatenate()([emb_make, emb_model, emb_variant, emb_fuel, norm_num])

# --- Dense Head ---
x = layers.Dense(128, activation='relu')(merged)
x = layers.Dropout(0.2)(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.1)(x)


output = layers.Dense(1, activation='linear', name='log_price_out')(x)

model = Model(inputs=[in_make, in_model, in_variant, in_fuel, in_num], outputs=output)

# Using Mean Squared Error for Log-Space regression is standard
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mean_absolute_error'])

--- Building Model ---


In [12]:




# ==========================================
# 4. TRAINING
# ==========================================
print("--- Starting Training ---")
history = model.fit(
    x=train_inputs,
    y=train_targets_log, # Training on Log Targets!
    validation_data=(val_inputs, val_targets_log),
    epochs=50, # Set high, let EarlyStopping handle it
    batch_size=512, # Large batch as requested
    callbacks=[
        callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)
    ],
    verbose=1
)

# ==========================================
# 5. EVALUATION (undoing the log)
# ==========================================
print("\n--- Final Evaluation ---")

# 1. Predict (Output is in Log Space)
log_preds = model.predict(test_inputs, batch_size=512)

# 2. Inverse Transform (Log -> GBP)
# expm1 reverses log1p
gbp_preds = np.expm1(log_preds).flatten()

# 3. Calculate Error in Real Money
mae = np.mean(np.abs(test_targets_raw - gbp_preds))
mape = np.mean(np.abs((test_targets_raw - gbp_preds) / test_targets_raw)) * 100

print(f"Final MAE: £{mae:,.2f}")
print(f"Final MAPE: {mape:.2f}% (Average percentage error)")

# 4. Show a few examples
results = pd.DataFrame({'Actual': test_targets_raw, 'Predicted': gbp_preds})
results['Error'] = results['Actual'] - results['Predicted']
print("\n--- Sample Predictions ---")
print(results.head(10))

--- Starting Training ---
Epoch 1/50
1194/1194 ━━━━━━━━━━━━━━━━━━━━ 23s 12ms/step - loss: 2.7629 - mean_absolute_error: 0.9037 - val_loss: 0.0444 - val_mean_absolute_error: 0.1482 - learning_rate: 0.0010
Epoch 2/50
1194/1194 ━━━━━━━━━━━━━━━━━━━━ 13s 10ms/step - loss: 0.5035 - mean_absolute_error: 0.5634 - val_loss: 0.0394 - val_mean_absolute_error: 0.1444 - learning_rate: 0.0010
Epoch 3/50
1194/1194 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - loss: 0.4368 - mean_absolute_error: 0.5251 - val_loss: 0.0325 - val_mean_absolute_error: 0.1268 - learning_rate: 0.0010
Epoch 4/50
1194/1194 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - loss: 0.3539 - mean_absolute_error: 0.4710 - val_loss: 0.0309 - val_mean_absolute_error: 0.1246 - learning_rate: 0.0010
Epoch 5/50
1194/1194 ━━━━━━━━━━━━━━━━━━━━ 12s 10ms/step - loss: 0.2878 - mean_absolute_error: 0.4244 - val_loss: 0.0273 - val_mean_absolute_error: 0.1150 - learning_rate: 0.0010
Epoch 6/50
1194/1194 ━━━━━━━━━━━━━━━━━━━━ 12s 10ms/step - loss: 0.2544 - mean_absolu

In [13]:
model.save('allcarmodel.keras')

In [8]:
model = tf.keras.models.load_model('allcarmodel.keras')

In [4]:
encoder = pickle.load(open('encoder.pkl', 'rb'))

In [12]:
single_input = encoder.transform_dict({'make': 'Ford', 'model': 'Fiesta', 'variant': 'Active', 'fuel_type': 'petrol', 'num_in': [2015, 50000]}) | {'num_in': np.array([[2015, 5000]])}

In [13]:
np.expm1(model.predict(single_input))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step


C:\Users\aevet\AppData\Local\Temp\ipykernel_18544\3963920109.py:1: RuntimeWarning: overflow encountered in expm1
  np.expm1(model.predict(single_input))


array([[inf]], dtype=float32)

In [14]:
model.predict(single_input)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step


array([[300.41794]], dtype=float32)